# Make animated plots of data - OHC

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from celluloid import Camera
from IPython.display import HTML
from matplotlib.colors import Normalize
from tqdm import tqdm

COUNTRIES_URL = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
WORLD = gpd.read_file(COUNTRIES_URL).rename({"ADMIN": "name"}, axis="columns")

## World map

In [ ]:
df = pd.read_csv("Data/ME4OH_EN411_OFAM3/SST_WORLD_1993_2014.csv")
# Convert dates to datetime, save year month and year indication columns (aids plotting)
df['Date'] = pd.to_datetime(df['Date'])
df['year_month'] = df['Date'].dt.to_period('M')
df['year'] = df['Date'].dt.to_period('Y')
df

In [ ]:
month_list = pd.date_range('1993-1-1','2014-12-31', freq='MS').strftime('%m/%Y').tolist()
year_list = pd.date_range('1993-1-1','2014-12-31', freq='YS').strftime('%Y').tolist()

In [ ]:
plt.rcParams.update({'font.size': 15})
fig, ax = plt.subplots(figsize=(16, 8))
camera = Camera(fig)

fig.tight_layout(pad=2.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_facecolor("lightblue")

ax.set_xlim([-180, 180])
ax.set_ylim([-90, 90])

# Create a shared normalization object
norm = Normalize(vmin= min(df.SST), vmax= max(df.SST))
cmap = plt.cm.plasma   # Define the colormap

for title, data in tqdm(df.groupby('year')):
    # Plot SST map
    WORLD.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    s = ax.scatter(data.Longitude, data.Latitude, c=data.SST, s=1, cmap='plasma', norm=norm)  # Plot SST observations
    ax.text(0.5, 1.01, title, transform=ax.transAxes)
    camera.snap()
    
plt.close()
animation = camera.animate()  # animation ready
HTML(animation.to_html5_video())

In [ ]:
animation.save('images/world_sst_year.mp4')